# TrkA_88 공개 설계의 AF2 재예측 · v0.3.0

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HisameOgasahara/tmp_protein/blob/main/TrkA_88_AF2_T4.ipynb?v=0.3.0)

T4 GPU에서 전체 실행하면 공개된 TrkA_88 서열과 RFdiffusion 설계 구조를 AF2에 입력합니다. 표적 구조를 템플릿으로 제공하고 설계 구조를 초기 좌표로 사용합니다. AF2 model_1_ptm, 단일 서열, recycle 3회, seed 0으로 실행합니다.

결과 표는 TrkA를 고정했을 때 결합체의 위치와 방향 차이를 비교합니다.

- 논문 보고값은 약 1.4 Å입니다.
- 공개 AF2 구조와 공개 설계 구조의 재계산값은 약 1.39 Å입니다.
- 새 AF2 예측을 원래 설계 및 공개 AF2 결과와 각각 비교합니다.

새 예측의 신뢰도 지표와 입력 자료, 예측 PDB, 설정, 패키지 목록, 실행 로그를 ZIP 하나에 저장합니다. 초기 설치에서 AF2 가중치 묶음을 내려받고 model_1_ptm 가중치를 추출하며, 같은 세션에서는 설치와 가중치를 재사용합니다.

[원 논문 그림 6](https://www.nature.com/articles/s41586-023-06415-8) · [서열·설계·예측 공개 자료](https://doi.org/10.6084/m9.figshare.22662094.v1) · [AF2 실행 방법 출처](https://github.com/sokrypton/ColabDesign/blob/27b15b11e39e0b99383975895bca9e0b0e907da1/colabdesign/rf/designability_test.py)


In [ ]:
from pathlib import Path
from tempfile import mkdtemp
import subprocess
import sys

checkout = Path(mkdtemp(prefix="trka88_main_", dir="/content")) / "repo"
clone = subprocess.run(
    ["git", "clone", "--depth", "1", "--single-branch", "--branch", "main",
     "https://github.com/HisameOgasahara/tmp_protein.git", str(checkout)],
    capture_output=True, text=True,
)
if clone.returncode:
    print(clone.stderr)
    clone.check_returncode()
revision = subprocess.check_output(["git", "-C", str(checkout), "rev-parse", "HEAD"], text=True).strip()
print(f"main: {revision[:12]}")


In [ ]:
import json
import pandas as pd
from IPython.display import display

log = checkout.parent / "trka88_workflow.log"
result_zip = None
with log.open("w", encoding="utf-8") as handle:
    with subprocess.Popen(
        [sys.executable, "-u", str(checkout / "trka_pipeline/workflow.py"), revision],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, encoding="utf-8",
    ) as process:
        for line in process.stdout:
            handle.write(line)
            handle.flush()
            print(line, end="", flush=True)
            if line.strip().endswith(".zip") and not line.startswith("Failure archive:"):
                result_zip = Path(line.strip())
        if process.wait():
            raise RuntimeError(f"AF2 failed. Full log: {log}")
if result_zip is None or not result_zip.is_file():
    raise FileNotFoundError(f"Missing result ZIP. Full log: {log}")
report = json.loads((result_zip.with_suffix("") / "comparison.json").read_text())
display(pd.DataFrame([
    {"비교": "논문 보고값", "표적 기준 결합체 RMSD (Å)": report["paper_pose_rmsd_angstrom"]},
    {"비교": "공개 AF2 vs 설계", "표적 기준 결합체 RMSD (Å)": report["published_af2_vs_design_angstrom"]},
    {"비교": "새 AF2 vs 설계", "표적 기준 결합체 RMSD (Å)": report["reprediction_vs_design_angstrom"]},
    {"비교": "새 AF2 vs 공개 AF2", "표적 기준 결합체 RMSD (Å)": report["reprediction_vs_published_af2_angstrom"]},
]).round(3))
print({key: report[key] for key in ("plddt", "iptm", "interface_pae_angstrom")})


In [ ]:
from google.colab import files
files.download(str(result_zip))
